# M5 Forecasting - Accuracy: EDA分析

## コンペティション概要
- **URL**: https://www.kaggle.com/competitions/m5-forecasting-accuracy
- **目的**: Walmart の商品販売数を28日間予測する
- **評価指標**: WRMSSE (Weighted Root Mean Squared Scaled Error)
- **データ**: 3州・10店舗・3,049商品の日次販売データ（約5.4年分）

## 大容量データ処理戦略
salesデータは30,490行×1,947列のワイド形式。全体をmeltすると約5,900万行に膨張するため、**meltせずにNumpy集計**する戦略を採用。

| 戦略 | 方法 | 用途 |
|------|------|------|
| Tier 1 | ワイド形式のままNumpy集計 | 大半のEDA分析 |
| Tier 2 | サブセット（1店舗等）のみmelt | 深堀り分析 |
| Tier 3 | 全体melt→Parquet保存 | 特徴量設計時（EDAでは不要） |

## 1. セットアップとデータ読み込み

In [4]:
import sys
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib
import seaborn as sns
'''
# --- Google Colab セットアップ ---
from google.colab import drive
drive.mount('/content/drive')
'''

# 日本語フォント設定
!apt-get -qq install fonts-ipafont
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
plt.rcParams['font.family'] = 'IPAGothic'

# --- ユーティリティ関数（src/utils から移植） ---
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f'Memory usage reduced from {start_mem:.2f} MB to {end_mem:.2f} MB '
              f'({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# --- 設定 ---
warnings.filterwarnings('ignore')
seed_everything(42)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# --- パス設定（Google Drive） ---
DRIVE_BASE = Path('/content/drive/MyDrive/Colab Data/m5-forecasting-accuracy')
INPUT_DIR = DRIVE_BASE / 'input'
OUTPUT_DIR = DRIVE_BASE / 'output'
INTERMEDIATE_DIR = DRIVE_BASE / 'intermediate'

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

'apt-get' �́A�����R�}���h�܂��͊O���R�}���h�A
����\�ȃv���O�����܂��̓o�b�` �t�@�C���Ƃ��ĔF������Ă��܂���B


FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf'

In [2]:
# --- カレンダーデータ（軽量） ---
calendar = pd.read_csv(INPUT_DIR / 'calendar.csv', parse_dates=['date'])
print(f'calendar: {calendar.shape}')

# --- 販売データ（メモリ最適化読み込み） ---
# dtype指定でピークメモリを451MB→約120MBに削減
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [f'd_{i}' for i in range(1, 1942)]

dtype_dict = {col: 'int16' for col in d_cols}
dtype_dict.update({col: 'category' for col in id_cols})

sales = pd.read_csv(INPUT_DIR / 'sales_train_evaluation.csv', dtype=dtype_dict)
print(f'sales: {sales.shape}')

# --- 価格データ（reduce_mem_usage適用） ---
sell_prices = pd.read_csv(INPUT_DIR / 'sell_prices.csv')
sell_prices = reduce_mem_usage(sell_prices)
print(f'sell_prices: {sell_prices.shape}')

# --- 提出サンプル ---
sample_sub = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
print(f'sample_submission: {sample_sub.shape}')

NameError: name 'INPUT_DIR' is not defined

In [ ]:
# メモリ使用量の確認
print('=== メモリ使用量 ===')
for name, df in [('calendar', calendar), ('sales', sales), 
                  ('sell_prices', sell_prices), ('sample_sub', sample_sub)]:
    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f'{name:15s}: {mem:8.1f} MB  ({df.shape[0]:>10,} rows × {df.shape[1]:>5,} cols)')

# Numpy行列を事前抽出（以降のEDAで繰り返し使用）
sales_matrix = sales[d_cols].values  # shape: (30490, 1941), dtype: int16
print(f'\nsales_matrix shape: {sales_matrix.shape}, dtype: {sales_matrix.dtype}')
print(f'sales_matrix memory: {sales_matrix.nbytes / 1024**2:.1f} MB')

In [ ]:
# 各テーブルの先頭行確認
print('=== calendar ===')
display(calendar.head(3))

print('\n=== sales (先頭6列 + 最初/最後のd列) ===')
display(sales[id_cols + ['d_1', 'd_2', 'd_3', 'd_1939', 'd_1940', 'd_1941']].head(3))

print('\n=== sell_prices ===')
display(sell_prices.head(3))

print('\n=== sample_submission ===')
display(sample_sub.head(3))

## 2. データ構造と階層構造の確認

M5データの階層構造:
```
State (3) → Store (10) → Category (3) → Department (7) → Item (~3,049/store)
```

In [ ]:
# 階層構造のユニーク数
print('=== 階層構造 ===')
for col in ['state_id', 'store_id', 'cat_id', 'dept_id', 'item_id']:
    n = sales[col].nunique()
    print(f'{col:12s}: {n:>6,} ユニーク値')

print(f'\n合計商品×店舗の組み合わせ（行数）: {len(sales):,}')

# 各レベルの内訳
print('\n=== 州別の店舗数 ===')
print(sales.groupby('state_id')['store_id'].nunique().to_string())

print('\n=== カテゴリ別の部門数 ===')
print(sales.groupby('cat_id')['dept_id'].nunique().to_string())

print('\n=== 部門別の商品数 ===')
print(sales.groupby('dept_id')['item_id'].nunique().sort_values(ascending=False).to_string())

In [ ]:
# 階層構造の可視化（ツリーマップ風テーブル）
hierarchy = (sales.groupby(['state_id', 'store_id', 'cat_id', 'dept_id'])
             .size().reset_index(name='item_count'))

fig, ax = plt.subplots(figsize=(12, 5))
pivot = hierarchy.pivot_table(index='store_id', columns='dept_id', 
                               values='item_count', fill_value=0).astype(int)
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('店舗×部門ごとの商品数')
ax.set_xlabel('部門')
ax.set_ylabel('店舗')
plt.tight_layout()
plt.show()

### 2.1 結合キーの確認

テーブル間の結合関係:
- `calendar.d` ↔ `sales` の `d_*` 列（日番号）
- `calendar.wm_yr_wk` ↔ `sell_prices.wm_yr_wk`（Walmart週番号）
- `(store_id, item_id)` が sales と sell_prices の共通キー

In [ ]:
# 結合キーの一致確認
print('=== 結合キー確認 ===')

# calendar.d と sales の d_* 列
cal_d_nums = set(calendar['d'].str[2:].astype(int))
sales_d_nums = set(range(1, 1942))
print(f'calendar日数: {len(cal_d_nums)}, sales日数: {len(sales_d_nums)}')
print(f'salesにあってcalendarにない日: {sales_d_nums - cal_d_nums}')
print(f'calendarにあってsalesにない日（予測期間）: {sorted(cal_d_nums - sales_d_nums)}')

# wm_yr_wk の一致
cal_weeks = set(calendar['wm_yr_wk'].unique())
price_weeks = set(sell_prices['wm_yr_wk'].unique())
print(f'\ncalendar週数: {len(cal_weeks)}, sell_prices週数: {len(price_weeks)}')
print(f'sell_pricesにあってcalendarにない週: {len(price_weeks - cal_weeks)}')

# store_id × item_id の一致
sales_keys = set(zip(sales['store_id'], sales['item_id']))
price_keys = set(zip(sell_prices['store_id'], sell_prices['item_id']))
print(f'\nsales (store,item)組: {len(sales_keys):,}')
print(f'sell_prices (store,item)組: {len(price_keys):,}')
print(f'salesにあってpricesにない: {len(sales_keys - price_keys)}')
print(f'pricesにあってsalesにない: {len(price_keys - sales_keys)}')

### 2.2 欠損データの分析

In [ ]:
# 各テーブルの欠損データ
print('=== calendar 欠損 ===')
cal_null = calendar.isnull().sum()
print(cal_null[cal_null > 0].to_string())
print(f'\n→ event列のNaN = イベントなし（構造的欠損）')

print('\n=== sell_prices 欠損 ===')
print(sell_prices.isnull().sum().to_string())
print(f'→ 全列完全（ただし、商品が販売開始前の週はレコード自体が存在しない）')

print('\n=== sales 欠損（d列） ===')
sales_null_count = np.isnan(sales_matrix.astype(float)).sum()
print(f'NaN数: {sales_null_count:,}')
print(f'→ 販売数は全て整数値で欠損なし（販売なし = 0）')

# 販売データのゼロ値の割合
zero_count = (sales_matrix == 0).sum()
total_count = sales_matrix.size
print(f'\nゼロ販売セル: {zero_count:,} / {total_count:,} ({100*zero_count/total_count:.1f}%)')
print('→ 大部分がゼロ（スパースデータ）')

In [ ]:
# sell_prices: 商品の販売開始時期を確認
# wm_yr_wkが小さいほど古い → 各(store,item)の最小weekが販売開始時期
price_start = sell_prices.groupby(['store_id', 'item_id'])['wm_yr_wk'].min()
week_range = calendar[['wm_yr_wk']].drop_duplicates().sort_values('wm_yr_wk')
min_week = week_range['wm_yr_wk'].min()
max_week = week_range['wm_yr_wk'].max()

print(f'calendar週範囲: {min_week} ~ {max_week}')
print(f'全{len(week_range)}週のうち、')
print(f'  最初の週から価格がある商品: {(price_start == min_week).sum():,} / {len(price_start):,}')
print(f'  途中から価格が出現する商品: {(price_start > min_week).sum():,} / {len(price_start):,}')

fig, ax = plt.subplots(figsize=(12, 4))
price_start.hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('商品の販売開始週（wm_yr_wk）の分布')
ax.set_xlabel('wm_yr_wk')
ax.set_ylabel('商品数')
plt.tight_layout()
plt.show()

## 3. 目的変数（販売数）の分布分析

In [ ]:
# 全販売数の基本統計量
all_sales = sales_matrix.flatten()
nonzero_sales = all_sales[all_sales > 0]

print('=== 全販売数の統計 ===')
print(f'全セル数:       {len(all_sales):>15,}')
print(f'ゼロ:           {(all_sales == 0).sum():>15,} ({100*(all_sales==0).mean():.1f}%)')
print(f'非ゼロ:         {len(nonzero_sales):>15,} ({100*len(nonzero_sales)/len(all_sales):.1f}%)')
print(f'平均（全体）:   {all_sales.mean():>15.2f}')
print(f'平均（非ゼロ）: {nonzero_sales.mean():>15.2f}')
print(f'中央値（全体）: {np.median(all_sales):>15.0f}')
print(f'中央値（非ゼロ）:{np.median(nonzero_sales):>15.0f}')
print(f'最大値:         {all_sales.max():>15,}')
print(f'99%ile:         {np.percentile(all_sales, 99):>15.0f}')
print(f'99.9%ile:       {np.percentile(all_sales, 99.9):>15.0f}')

del all_sales  # メモリ解放

In [ ]:
# 販売数の分布（非ゼロのみ）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 非ゼロ販売数のヒストグラム（上限クリップ）
axes[0].hist(nonzero_sales[nonzero_sales <= 20], bins=20, 
             color='steelblue', edgecolor='white', density=True)
axes[0].axvline(np.median(nonzero_sales), color='red', ls='--', label=f'中央値={np.median(nonzero_sales):.0f}')
axes[0].axvline(nonzero_sales.mean(), color='orange', ls='--', label=f'平均={nonzero_sales.mean():.1f}')
axes[0].set_title('非ゼロ販売数の分布（≤20）')
axes[0].set_xlabel('販売数')
axes[0].set_ylabel('密度')
axes[0].legend()

# 右: log1p変換後の分布
log_sales = np.log1p(nonzero_sales)
axes[1].hist(log_sales, bins=50, color='coral', edgecolor='white', density=True)
axes[1].set_title('非ゼロ販売数のlog1p変換後分布')
axes[1].set_xlabel('log1p(販売数)')
axes[1].set_ylabel('密度')

plt.tight_layout()
plt.show()

del nonzero_sales, log_sales

In [ ]:
# 商品ごとのゼロ販売率分布
zero_ratio_per_item = (sales_matrix == 0).mean(axis=1)
mean_sales_per_item = sales_matrix.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(zero_ratio_per_item, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(zero_ratio_per_item.mean(), color='red', ls='--',
                label=f'平均={zero_ratio_per_item.mean():.2f}')
axes[0].set_title('商品ごとのゼロ販売率分布')
axes[0].set_xlabel('ゼロ販売率')
axes[0].set_ylabel('商品数')
axes[0].legend()

axes[1].hist(mean_sales_per_item[mean_sales_per_item <= 5], bins=50, 
             color='coral', edgecolor='white')
axes[1].set_title('商品ごとの平均販売数分布（≤5）')
axes[1].set_xlabel('平均販売数/日')
axes[1].set_ylabel('商品数')

plt.tight_layout()
plt.show()

# 間歇需要（intermittent demand）の割合
print(f'ゼロ販売率 > 90% の商品: {(zero_ratio_per_item > 0.9).sum():,} ({100*(zero_ratio_per_item > 0.9).mean():.1f}%)')
print(f'ゼロ販売率 > 50% の商品: {(zero_ratio_per_item > 0.5).sum():,} ({100*(zero_ratio_per_item > 0.5).mean():.1f}%)')

## 4. 時系列パターン分析

Tier 1戦略: ワイド形式のままNumpy集計で日次トレンドを分析

In [ ]:
# calendarにd番号を追加（Numpy集計結果との紐付け用）
calendar['d_num'] = calendar['d'].str[2:].astype(int)

# 日次全体販売数トレンド
daily_total = sales_matrix.sum(axis=0)  # shape: (1941,)
dates = calendar[calendar['d_num'].between(1, 1941)].sort_values('d_num')['date'].values

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(dates, daily_total, linewidth=0.5, color='steelblue', alpha=0.8)
ax.set_title('日次合計販売数の推移')
ax.set_xlabel('日付')
ax.set_ylabel('合計販売数')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# カテゴリ別の日次販売トレンド
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
colors = {'FOODS': 'green', 'HOBBIES': 'orange', 'HOUSEHOLD': 'purple'}

for ax, cat in zip(axes, sales['cat_id'].cat.categories):
    mask = (sales['cat_id'] == cat).values
    cat_daily = sales_matrix[mask].sum(axis=0)
    ax.plot(dates, cat_daily, linewidth=0.5, color=colors.get(cat, 'steelblue'), alpha=0.8)
    ax.set_title(f'{cat} - 日次販売数')
    ax.set_ylabel('販売数')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

axes[-1].set_xlabel('日付')
plt.tight_layout()
plt.show()

In [ ]:
# 州別の日次販売トレンド
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
state_colors = {'CA': 'royalblue', 'TX': 'tomato', 'WI': 'forestgreen'}

for ax, state in zip(axes, sales['state_id'].cat.categories):
    mask = (sales['state_id'] == state).values
    state_daily = sales_matrix[mask].sum(axis=0)
    ax.plot(dates, state_daily, linewidth=0.5, color=state_colors.get(state, 'gray'), alpha=0.8)
    ax.set_title(f'{state} - 日次販売数')
    ax.set_ylabel('販売数')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

axes[-1].set_xlabel('日付')
plt.tight_layout()
plt.show()

In [ ]:
# 曜日別・月別パターン
cal_subset = calendar[calendar['d_num'].between(1, 1941)].sort_values('d_num').copy()
cal_subset['daily_total'] = daily_total

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 曜日別
dow_order = ['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
dow_sales = cal_subset.groupby('weekday')['daily_total'].mean().reindex(dow_order)
axes[0].bar(range(7), dow_sales.values, color='steelblue', edgecolor='white')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(['土', '日', '月', '火', '水', '木', '金'])
axes[0].set_title('曜日別の平均販売数')
axes[0].set_ylabel('平均合計販売数')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

# 月別
month_sales = cal_subset.groupby('month')['daily_total'].mean()
axes[1].bar(month_sales.index, month_sales.values, color='coral', edgecolor='white')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels([f'{m}月' for m in range(1, 13)])
axes[1].set_title('月別の平均販売数')
axes[1].set_ylabel('平均合計販売数')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# 曜日×月のヒートマップ
dow_month = cal_subset.groupby(['weekday', 'month'])['daily_total'].mean().unstack()
dow_month = dow_month.reindex(dow_order)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(dow_month, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            yticklabels=['土', '日', '月', '火', '水', '木', '金'],
            xticklabels=[f'{m}月' for m in range(1, 13)])
ax.set_title('曜日×月の平均合計販売数')
plt.tight_layout()
plt.show()

In [ ]:
# 店舗別の平均日次販売数
store_daily_mean = []
for store in sales['store_id'].cat.categories:
    mask = (sales['store_id'] == store).values
    store_daily_mean.append({
        'store_id': store,
        'mean_daily_sales': sales_matrix[mask].sum(axis=0).mean(),
        'item_count': mask.sum()
    })
store_df = pd.DataFrame(store_daily_mean).sort_values('mean_daily_sales', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(store_df['store_id'], store_df['mean_daily_sales'], 
              color=[state_colors.get(s[:2], 'gray') for s in store_df['store_id']],
              edgecolor='white')
ax.set_title('店舗別の平均日次販売数')
ax.set_xlabel('店舗')
ax.set_ylabel('平均日次販売数')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

# 州の凡例
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in state_colors.items()]
ax.legend(handles=legend_elements, title='州')
plt.tight_layout()
plt.show()

## 5. イベント・祝日・SNAPの影響分析

In [ ]:
# イベント情報の概要
print('=== イベント情報 ===')
print(f'イベントがある日数: {calendar["event_name_1"].notna().sum()}')
print(f'  うち2つ目のイベントもある日: {calendar["event_name_2"].notna().sum()}')

print('\n=== イベントタイプ別の日数 ===')
print(calendar['event_type_1'].value_counts().to_string())

print('\n=== 主要イベント名（上位10） ===')
print(calendar['event_name_1'].value_counts().head(10).to_string())

In [ ]:
# イベント日 vs 非イベント日の販売数比較
cal_subset['has_event'] = cal_subset['event_name_1'].notna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# イベント有無の比較
event_group = cal_subset.groupby('has_event')['daily_total'].mean()
axes[0].bar(['イベントなし', 'イベントあり'], event_group.values, 
            color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('イベント有無による平均販売数')
axes[0].set_ylabel('平均合計販売数')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
uplift = (event_group[True] / event_group[False] - 1) * 100
axes[0].text(1, event_group[True] * 0.95, f'+{uplift:.1f}%', ha='center', fontsize=12, fontweight='bold')

# イベントタイプ別
event_type_sales = cal_subset[cal_subset['has_event']].groupby('event_type_1')['daily_total'].mean().sort_values()
axes[1].barh(event_type_sales.index, event_type_sales.values, color='coral', edgecolor='white')
axes[1].set_title('イベントタイプ別の平均販売数')
axes[1].set_xlabel('平均合計販売数')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# SNAP（補助栄養プログラム）の影響分析（州別）
# SNAPは州ごとに異なる日に適用される
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, state in zip(axes, ['CA', 'TX', 'WI']):
    snap_col = f'snap_{state}'
    mask = (sales['state_id'] == state).values
    state_daily = sales_matrix[mask].sum(axis=0)
    
    cal_state = cal_subset.copy()
    cal_state['state_daily'] = state_daily
    
    snap_group = cal_state.groupby(snap_col)['state_daily'].mean()
    ax.bar(['SNAP なし', 'SNAP あり'], snap_group.values, 
           color=[state_colors[state], 'gold'], edgecolor='white')
    ax.set_title(f'{state} - SNAP有無の販売数')
    ax.set_ylabel('平均販売数')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))
    
    if len(snap_group) == 2:
        uplift = (snap_group[1] / snap_group[0] - 1) * 100
        ax.text(1, snap_group[1] * 0.95, f'+{uplift:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. 価格データの分析

In [ ]:
# 価格の基本統計
print('=== 価格データの基本統計 ===')
print(sell_prices['sell_price'].describe().to_string())

# カテゴリ情報をitem_idから抽出
sell_prices['cat_id'] = sell_prices['item_id'].str.rsplit('_', n=2).str[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# カテゴリ別の価格分布
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    cat_prices = sell_prices[sell_prices['cat_id'] == cat]['sell_price']
    axes[0].hist(cat_prices[cat_prices <= 30], bins=60, alpha=0.5, label=cat, density=True)
axes[0].set_title('カテゴリ別の価格分布（≤$30）')
axes[0].set_xlabel('価格 ($)')
axes[0].set_ylabel('密度')
axes[0].legend()

# 価格変動の分析（商品ごとの価格変動係数）
price_cv = sell_prices.groupby(['store_id', 'item_id'])['sell_price'].agg(['mean', 'std'])
price_cv['cv'] = price_cv['std'] / price_cv['mean']
price_cv = price_cv.dropna()

axes[1].hist(price_cv['cv'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('商品ごとの価格変動係数（CV）分布')
axes[1].set_xlabel('変動係数')
axes[1].set_ylabel('商品数')

plt.tight_layout()
plt.show()

print(f'\n価格変動なし（CV=0）の商品: {(price_cv["cv"] == 0).sum():,} / {len(price_cv):,} ({100*(price_cv["cv"]==0).mean():.1f}%)')
print(f'価格変動あり（CV>0）の商品: {(price_cv["cv"] > 0).sum():,} / {len(price_cv):,}')

In [ ]:
# 価格の時系列トレンド（カテゴリ別平均価格）
price_weekly = sell_prices.groupby(['wm_yr_wk', 'cat_id'])['sell_price'].mean().reset_index()
week_to_date = calendar[['wm_yr_wk', 'date']].drop_duplicates('wm_yr_wk').sort_values('wm_yr_wk')
price_weekly = price_weekly.merge(week_to_date, on='wm_yr_wk')

fig, ax = plt.subplots(figsize=(14, 5))
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    cat_data = price_weekly[price_weekly['cat_id'] == cat]
    ax.plot(cat_data['date'], cat_data['sell_price'], label=cat, alpha=0.8)

ax.set_title('カテゴリ別の平均価格推移（週次）')
ax.set_xlabel('日付')
ax.set_ylabel('平均価格 ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. 階層別売上構造の分析

In [ ]:
# 部門別の合計販売数と構成比
dept_total = []
for dept in sales['dept_id'].cat.categories:
    mask = (sales['dept_id'] == dept).values
    dept_total.append({
        'dept_id': dept,
        'total_sales': sales_matrix[mask].sum(),
        'item_count': mask.sum(),
        'mean_per_item_day': sales_matrix[mask].mean()
    })
dept_df = pd.DataFrame(dept_total).sort_values('total_sales', ascending=False)
dept_df['share'] = 100 * dept_df['total_sales'] / dept_df['total_sales'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 部門別の合計販売数
axes[0].barh(dept_df['dept_id'], dept_df['total_sales'], color='steelblue', edgecolor='white')
axes[0].set_title('部門別の合計販売数')
axes[0].set_xlabel('合計販売数')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1e6:.0f}M'))

# 構成比の円グラフ
axes[1].pie(dept_df['share'], labels=dept_df['dept_id'], autopct='%.1f%%', startangle=90)
axes[1].set_title('部門別の売上構成比')

plt.tight_layout()
plt.show()

print('=== 部門別詳細 ===')
print(dept_df.to_string(index=False))

In [ ]:
# 店舗×部門の販売数ヒートマップ
store_dept_sales = []
for store in sales['store_id'].cat.categories:
    for dept in sales['dept_id'].cat.categories:
        mask = ((sales['store_id'] == store) & (sales['dept_id'] == dept)).values
        if mask.sum() > 0:
            store_dept_sales.append({
                'store_id': store, 'dept_id': dept,
                'mean_daily': sales_matrix[mask].sum(axis=0).mean()
            })

sd_df = pd.DataFrame(store_dept_sales)
sd_pivot = sd_df.pivot(index='store_id', columns='dept_id', values='mean_daily')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(sd_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('店舗×部門の平均日次販売数')
ax.set_xlabel('部門')
ax.set_ylabel('店舗')
plt.tight_layout()
plt.show()

## 8. 学習/検証/評価期間の比較

- **学習期間**: d_1 ~ d_1913 (2011-01-29 ~ 2016-04-24)
- **検証期間（Public LB）**: d_1914 ~ d_1941 (2016-04-25 ~ 2016-05-22)
- **評価期間（Private LB）**: d_1942 ~ d_1969 (2016-05-23 ~ 2016-06-19, 予測対象)

In [ ]:
# 期間の定義
train_days = list(range(1, 1914))      # d_1 ~ d_1913
val_days = list(range(1914, 1942))     # d_1914 ~ d_1941（28日間）

# 各期間の日付範囲
cal_train = calendar[calendar['d_num'].isin(train_days)]
cal_val = calendar[calendar['d_num'].isin(val_days)]

print(f'学習期間: {cal_train["date"].min().date()} ~ {cal_train["date"].max().date()} ({len(train_days)}日)')
print(f'検証期間: {cal_val["date"].min().date()} ~ {cal_val["date"].max().date()} ({len(val_days)}日)')

# 曜日分布の比較
print('\n=== 曜日分布の比較 ===')
train_dow = cal_train['weekday'].value_counts(normalize=True).reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
val_dow = cal_val['weekday'].value_counts(normalize=True).reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

dow_compare = pd.DataFrame({'学習': train_dow, '検証': val_dow})
print(dow_compare.to_string())

# イベント分布の比較
print('\n=== イベント率の比較 ===')
print(f'学習期間のイベント日率: {cal_train["event_name_1"].notna().mean():.3f}')
print(f'検証期間のイベント日率: {cal_val["event_name_1"].notna().mean():.3f}')

# SNAP分布の比較
print('\n=== SNAP率の比較 ===')
for state in ['CA', 'TX', 'WI']:
    col = f'snap_{state}'
    print(f'{state}: 学習={cal_train[col].mean():.3f}, 検証={cal_val[col].mean():.3f}')

In [ ]:
# 検証期間の販売数分布 vs 学習期間の直近28日
train_last28 = sales_matrix[:, -56:-28]  # 学習期間の最後28日
val_period = sales_matrix[:, -28:]        # 検証期間

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 日次合計の比較
train_last28_daily = train_last28.sum(axis=0)
val_daily = val_period.sum(axis=0)

axes[0].plot(range(28), train_last28_daily, label='学習期間（直近28日）', color='steelblue')
axes[0].plot(range(28), val_daily, label='検証期間（28日）', color='coral')
axes[0].set_title('直近28日間の日次販売数比較')
axes[0].set_xlabel('日数')
axes[0].set_ylabel('合計販売数')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

# 商品ごとの平均販売数の比較
mean_train = train_last28.mean(axis=1)
mean_val = val_period.mean(axis=1)
axes[1].scatter(mean_train, mean_val, alpha=0.1, s=1)
max_val = max(mean_train.max(), mean_val.max())
axes[1].plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
axes[1].set_title('商品別平均販売数: 学習直近28日 vs 検証期間')
axes[1].set_xlabel('学習期間（直近28日）')
axes[1].set_ylabel('検証期間')
axes[1].legend()
axes[1].set_xlim(0, 15)
axes[1].set_ylim(0, 15)

plt.tight_layout()
plt.show()

corr = np.corrcoef(mean_train, mean_val)[0, 1]
print(f'商品別平均販売数の相関係数: {corr:.4f}')

## 9. 月次×カテゴリの需要・価格構造分析

特徴量設計の前に、**「何がいつ効くのか」**を月単位で把握する。
特に、価格上昇と販売数増加が同時に起こる時期（通常の価格弾力性が崩れる時期）を特定し、
Rolling特徴量の窓幅設計やモデル分割の根拠とする。

アメリカ小売の主な需要イベント:
- **Tax Refund Season** (2-4月): 還付金で消費が活性化
- **Back to School** (8-9月): 学用品需要
- **Thanksgiving〜Christmas** (11-12月): 年末商戦
- **SNAP給付**: 月初に集中、州ごとに日程が異なる

In [ ]:
# --- セル①: 月次×カテゴリの平均日次販売数ヒートマップ ---

# 年-月ラベルを作成
cal_subset['year_month'] = cal_subset['date'].dt.to_period('M')

# カテゴリ別の日次販売数を cal_subset に追加（まだない場合）
for cat in sales['cat_id'].cat.categories:
    col_name = f'daily_{cat}'
    if col_name not in cal_subset.columns:
        mask = (sales['cat_id'] == cat).values
        cal_subset[col_name] = sales_matrix[mask].sum(axis=0)

# 月次×カテゴリ集計
monthly_cat = cal_subset.groupby('year_month').agg(
    daily_total_mean=('daily_total', 'mean'),
    FOODS_mean=('daily_FOODS', 'mean'),
    HOBBIES_mean=('daily_HOBBIES', 'mean'),
    HOUSEHOLD_mean=('daily_HOUSEHOLD', 'mean'),
).reset_index()
monthly_cat['year_month_str'] = monthly_cat['year_month'].astype(str)

# ヒートマップ用に整形
heatmap_data = monthly_cat.set_index('year_month_str')[
    ['FOODS_mean', 'HOBBIES_mean', 'HOUSEHOLD_mean']
].T
heatmap_data.index = ['FOODS', 'HOBBIES', 'HOUSEHOLD']

fig, ax = plt.subplots(figsize=(20, 4))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, ax=ax, 
            linewidths=0.5, linecolor='white')
ax.set_title('月次×カテゴリ 平均日次販売数（色が濃い＝需要が高い）')
ax.set_xlabel('年-月')
ax.set_ylabel('カテゴリ')
# x軸ラベルを間引き（6ヶ月ごと）
tick_positions = list(range(0, len(heatmap_data.columns), 6))
ax.set_xticks([p + 0.5 for p in tick_positions])
ax.set_xticklabels([heatmap_data.columns[p] for p in tick_positions], rotation=45)
plt.tight_layout()
plt.show()

# 月別（1-12月）平均を折れ線で比較
monthly_pattern = cal_subset.groupby('month').agg(
    FOODS=('daily_FOODS', 'mean'),
    HOBBIES=('daily_HOBBIES', 'mean'),
    HOUSEHOLD=('daily_HOUSEHOLD', 'mean'),
)

fig, ax = plt.subplots(figsize=(10, 5))
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    vals = monthly_pattern[cat]
    # 各カテゴリを自身の平均で正規化（スケール差を吸収）
    ax.plot(range(1, 13), vals / vals.mean(), marker='o', label=cat)
ax.set_xticks(range(1, 13))
ax.set_xticklabels([f'{m}月' for m in range(1, 13)])
ax.set_title('月別の販売数パターン（各カテゴリの平均=1.0に正規化）')
ax.set_ylabel('正規化販売数')
ax.axhline(1.0, color='gray', ls='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- セル②: 月次×カテゴリの価格-販売数相関 ---
# 週次価格データを月次に変換し、販売数との相関を月ごとに算出

# sell_prices に月情報を付与
week_to_month = calendar[['wm_yr_wk', 'year', 'month']].drop_duplicates('wm_yr_wk')
sp = sell_prices.merge(week_to_month, on='wm_yr_wk')
sp['cat_id'] = sp['item_id'].str.rsplit('_', n=2).str[0]

# 月次×カテゴリの平均価格
monthly_price = sp.groupby(['year', 'month', 'cat_id'])['sell_price'].mean().reset_index()
monthly_price['year_month'] = monthly_price['year'].astype(str) + '-' + monthly_price['month'].astype(str).str.zfill(2)

# 月次販売数（cal_subsetから）
monthly_sales_long = cal_subset.groupby(['year', 'month']).agg(
    FOODS=('daily_FOODS', 'mean'),
    HOBBIES=('daily_HOBBIES', 'mean'),
    HOUSEHOLD=('daily_HOUSEHOLD', 'mean'),
).reset_index()
monthly_sales_melted = monthly_sales_long.melt(
    id_vars=['year', 'month'], var_name='cat_id', value_name='mean_daily_sales'
)

# 結合
monthly_merged = monthly_price.merge(monthly_sales_melted, on=['year', 'month', 'cat_id'])

# カテゴリ別に月次相関を計算（12ヶ月ローリング）
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for ax, cat in zip(axes, ['FOODS', 'HOBBIES', 'HOUSEHOLD']):
    cat_data = monthly_merged[monthly_merged['cat_id'] == cat].sort_values('year_month').reset_index(drop=True)
    
    # 12ヶ月ローリング相関
    rolling_corr = cat_data['sell_price'].rolling(12).corr(cat_data['mean_daily_sales'])
    
    ax.plot(cat_data['year_month'], rolling_corr, color='steelblue', linewidth=1.5)
    ax.axhline(0, color='black', ls='-', linewidth=0.5)
    ax.fill_between(cat_data['year_month'], rolling_corr, 0,
                    where=(rolling_corr > 0), color='red', alpha=0.3, label='正の相関（価格↑販売↑）')
    ax.fill_between(cat_data['year_month'], rolling_corr, 0,
                    where=(rolling_corr <= 0), color='blue', alpha=0.3, label='負の相関（価格↑販売↓）')
    ax.set_title(f'{cat} - 価格と販売数の12ヶ月ローリング相関')
    ax.set_ylabel('相関係数')
    ax.set_ylim(-1, 1)
    ax.legend(loc='lower left')
    # x軸ラベルを間引き
    xticks = list(range(0, len(cat_data), 6))
    ax.set_xticks(xticks)
    ax.set_xticklabels([cat_data['year_month'].iloc[i] for i in xticks], rotation=45)

axes[-1].set_xlabel('年-月')
plt.suptitle('価格弾力性の時期別変動（正の相関 = 需要主導で価格も販売も上昇する時期）', y=1.02)
plt.tight_layout()
plt.show()

# 月別（1-12月）の平均相関を表で表示
print('=== 月別（1-12月）の価格-販売数 平均相関 ===')
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    cat_data = monthly_merged[monthly_merged['cat_id'] == cat]
    month_corr = cat_data.groupby('month').apply(
        lambda g: g['sell_price'].corr(g['mean_daily_sales'])
    )
    positive_months = month_corr[month_corr > 0].index.tolist()
    print(f'\n{cat}:')
    print(f'  正の相関が出る月: {positive_months if positive_months else "なし"}')
    print(f'  月別相関: {month_corr.round(3).to_dict()}')

In [ ]:
# --- セル③: 異常月のイベント・SNAP突合 ---
# 「正の相関が出る月」や「販売数が急増する月」に何が起きているかを特定

# 月別のイベント日数とSNAP日数を集計
monthly_context = cal_subset.groupby(['year', 'month']).agg(
    n_days=('date', 'count'),
    n_event_days=('has_event', 'sum'),
    snap_CA_days=('snap_CA', 'sum'),
    snap_TX_days=('snap_TX', 'sum'),
    snap_WI_days=('snap_WI', 'sum'),
    daily_total_mean=('daily_total', 'mean'),
).reset_index()
monthly_context['year_month'] = monthly_context['year'].astype(str) + '-' + monthly_context['month'].astype(str).str.zfill(2)

# 月別のイベント名一覧
event_by_month = cal_subset[cal_subset['has_event']].groupby(['year', 'month'])['event_name_1'].apply(
    lambda x: ', '.join(sorted(x.unique()))
).reset_index()
event_by_month.columns = ['year', 'month', 'events']
monthly_context = monthly_context.merge(event_by_month, on=['year', 'month'], how='left')
monthly_context['events'] = monthly_context['events'].fillna('')

# 販売数が前年同月比で大きく変動した月を検出
monthly_context['prev_year_sales'] = monthly_context.groupby('month')['daily_total_mean'].shift(1)
monthly_context['yoy_change_pct'] = (
    (monthly_context['daily_total_mean'] - monthly_context['prev_year_sales']) 
    / monthly_context['prev_year_sales'] * 100
)

# 前年同月比が大きい月（±10%以上）を表示
notable = monthly_context[monthly_context['yoy_change_pct'].abs() > 10].sort_values('yoy_change_pct', ascending=False)

print('=== 前年同月比 ±10%以上の月 ===')
print(f'{"年-月":>10s} {"前年比":>8s} {"イベント日数":>8s} {"SNAP(CA)":>8s} {"主なイベント"}')
print('-' * 80)
for _, row in notable.iterrows():
    print(f'{row["year_month"]:>10s} {row["yoy_change_pct"]:>+7.1f}% {int(row["n_event_days"]):>8d} '
          f'{int(row["snap_CA_days"]):>8d} {row["events"][:50]}')

# 月別の販売数成長率のヒートマップ
yoy_pivot = monthly_context.pivot_table(index='month', columns='year', values='yoy_change_pct')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(yoy_pivot, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax,
            yticklabels=[f'{m}月' for m in range(1, 13)])
ax.set_title('月別の販売数 前年同月比 (%)（赤=成長, 青=減少）')
ax.set_xlabel('年')
ax.set_ylabel('月')
plt.tight_layout()
plt.show()

print('\n=== 知見まとめ ===')
print('この表から、販売数の急増/急減がイベントやSNAPの変動で説明できるかを確認する。')
print('説明できない変動は、トレンド（店舗増加等）や外部要因の可能性がある。')

## 10. 特徴量候補のリストアップ

EDA分析から得られた知見に基づき、モデリングで使用する特徴量の候補をリストアップする。

In [ ]:
feature_candidates = {
    'カレンダー特徴量': [
        '曜日 (weekday/wday)',
        '月 (month)',
        '年 (year)',
        '月内日 (day of month)',
        '週番号 (week of year)',
        '週末フラグ (is_weekend)',
    ],
    'イベント特徴量': [
        'イベント有無フラグ (has_event)',
        'イベントタイプ (event_type)',
        'イベント前後の日数',
    ],
    'SNAP特徴量': [
        'SNAP適用フラグ (snap_CA/TX/WI)',
        'SNAP前後の日数',
    ],
    'ラグ特徴量': [
        '販売数ラグ (lag_7, lag_14, lag_28)',
        '販売数ラグ (lag_1は予測期間28日のため使用不可)',
    ],
    'ローリング特徴量': [
        '移動平均 (rolling_mean_7, 14, 28, 60)',
        '移動標準偏差 (rolling_std_7, 14, 28)',
        '移動最大/最小',
        'ゼロ販売日数の移動合計',
    ],
    '価格特徴量': [
        '現在価格 (sell_price)',
        '価格変化率 (price_change_pct)',
        '最大価格からの割引率',
        '価格の移動平均からの乖離',
        '価格変動係数 (price_cv)',
    ],
    '階層集計特徴量': [
        '店舗レベル日次合計/平均',
        '部門レベル日次合計/平均',
        'カテゴリレベル日次合計/平均',
        '州レベル日次合計/平均',
        '商品の店舗内シェア',
    ],
    'エンコーディング特徴量': [
        'store_id (Label Encoding)',
        'dept_id (Label Encoding)',
        'cat_id (Label Encoding)',
        'state_id (Label Encoding)',
        'item_id (Label Encoding)',
    ],
}

print('=== 特徴量候補一覧 ===')
total = 0
for category, features in feature_candidates.items():
    print(f'\n【{category}】')
    for f in features:
        print(f'  - {f}')
    total += len(features)
print(f'\n合計: {total} 特徴量候補')

## 11. EDAまとめ・中間データ保存

In [ ]:
import pickle

eda_results = {
    'data_info': {
        'n_items': len(sales),
        'n_days_train': 1913,
        'n_days_val': 28,
        'n_days_eval': 28,
        'n_stores': sales['store_id'].nunique(),
        'n_categories': sales['cat_id'].nunique(),
        'n_departments': sales['dept_id'].nunique(),
        'n_states': sales['state_id'].nunique(),
        'date_range_train': ('2011-01-29', '2016-04-24'),
        'date_range_val': ('2016-04-25', '2016-05-22'),
    },
    'target_stats': {
        'zero_ratio_overall': float((sales_matrix == 0).mean()),
        'mean_sales_overall': float(sales_matrix.mean()),
        'items_zero_ratio_gt90pct': int((zero_ratio_per_item > 0.9).sum()),
    },
    'feature_candidates': feature_candidates,
    'key_findings': [
        '販売データの大部分がゼロ（スパースデータ） → 間歇需要対応が重要',
        '曜日効果が顕著（土曜日が最も多い）',
        'FOODS_3部門が売上の最大シェア',
        'イベント日は販売数が増加する傾向',
        'SNAPは州によって影響度が異なる',
        '価格変動がある商品が多い → 価格特徴量が有効',
        '学習期間と検証期間の販売パターンは高い相関を示す',
        '商品の販売開始時期がバラバラ → 販売開始前のゼロは除外が必要',
    ],
    'memory_strategy': {
        'sales_dtype': 'int16 (category型ID列)',
        'peak_memory_mb': 120,
        'approach': 'meltせずNumpy集計（Tier 1戦略）',
    },
}

# 保存
INTERMEDIATE_DIR.mkdir(exist_ok=True)
with open(INTERMEDIATE_DIR / '01_eda_results.pkl', 'wb') as f:
    pickle.dump(eda_results, f)

print('01_eda_results.pkl を保存しました')
print(f'\n=== 主要な知見 ===')
for i, finding in enumerate(eda_results['key_findings'], 1):
    print(f'{i}. {finding}')

In [ ]:
# 日次集計データの保存（下流ノートブック用）
daily_agg = cal_subset[['date', 'd_num', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
                          'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI',
                          'daily_total', 'has_event']].copy()

# カテゴリ別・州別の日次合計を追加
for cat in sales['cat_id'].cat.categories:
    mask = (sales['cat_id'] == cat).values
    daily_agg[f'daily_{cat}'] = sales_matrix[mask].sum(axis=0)

for state in sales['state_id'].cat.categories:
    mask = (sales['state_id'] == state).values
    daily_agg[f'daily_{state}'] = sales_matrix[mask].sum(axis=0)

daily_agg.to_parquet(INTERMEDIATE_DIR / '01_sales_agg_daily.parquet', index=False)
print(f'01_sales_agg_daily.parquet を保存しました: {daily_agg.shape}')
print(f'ファイルサイズ: {(INTERMEDIATE_DIR / "01_sales_agg_daily.parquet").stat().st_size / 1024:.1f} KB')